<div align="center" style="font-size: 2.5em; font-weight: bold; margin-bottom: 10px;">Diplomski rad — Vanilla pristup</div>

<div align="center" style="font-size: 1.2em; color: #555;">Qwen2.5-14B-Instruct &middot; bez RAG konteksta</div>

<div align="right" style="font-style: italic; color: #666;">by Zlatko Pračić</div>

## 1. Postavljanje okruženja

Ovaj dio notebooka instalira sve potrebne biblioteke, montira Google Drive i konfigurira pristup Hugging Face platformi.

### Instalacija potrebnih biblioteka

Instaliraju se sve Python biblioteke potrebne za rad:
- **bitsandbytes** - kvantizacija modela za uštedu memorije
- **transformers, accelerate** - rad s LLM modelima

In [1]:
!pip install -q -U bitsandbytes transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 149.0 MB/s eta 0:00:00


---

**bitsandbytes**
https://huggingface.co/docs/transformers/en/quantization/bitsandbytes
https://pypi.org/project/bitsandbytes/

**transformers**
https://huggingface.co/docs/transformers/index
https://pypi.org/project/transformers/

**torch (PyTorch)**
https://pytorch.org/docs/stable/index.html
https://pypi.org/project/torch/

**accelerate**
https://huggingface.co/docs/accelerate/index
https://pypi.org/project/accelerate/

**sentencepiece**
https://github.com/google/sentencepiece
https://pypi.org/project/sentencepiece/

---

### Montiranje Google Drivea

Montira se Google Drive kako bi se moglo pristupiti dokumentima i podacima za diplomski rad.

### Uvoz biblioteka

Uvoze se svi potrebni moduli na jednom mjestu zbog preglednosti.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig, GenerationConfig
from google.colab import userdata
from huggingface_hub import login
import torch
import time
import pandas as pd
import os
import json
from copy import deepcopy
import re

### Autentifikacija s Hugging Face

In [3]:
token = userdata.get('HF_TOKEN')
login(token)

**Učitavanje modela**

### Odabir LLM modela

Odabire se model za generaciju teksta. Dostupne alternative navedene su u komentarima.

[Qwen2.5 model](https://huggingface.co/Qwen/Qwen2.5-14B-Instruct)

[EuroLLM-22B-Instruct-2512](https://huggingface.co/utter-project/EuroLLM-22B-Instruct-2512)

In [4]:
# ============================================================
# OPCIJA A: čitanje LLM-a izravno s Google Drivea
# Pokreni OVU ćeliju ILI ćeliju B — ne obje.
# ============================================================

# model_name = "Qwen/Qwen2.5-14B-Instruct"
model_name = "utter-project/EuroLLM-22B-Instruct-2512"

from google.colab import drive
drive.mount('/content/drive')

os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print(f"Model: {model_name}")
print(f"Čitanje s Drivea: {os.environ['HF_HOME']}")

Mounted at /content/drive
Model: utter-project/EuroLLM-22B-Instruct-2512
Čitanje s Drivea: /content/drive/MyDrive/hf_cache


In [ ]:
# ============================================================
# OPCIJA B: kopiranje LLM-a na /content (brži disk) pa čitanje
# Pokreni OVU ćeliju ILI ćeliju A — ne obje.
# ============================================================

import os

model_name = "Qwen/Qwen2.5-14B-Instruct"
# model_name = "utter-project/EuroLLM-22B-Instruct-2512"

from google.colab import drive
drive.mount('/content/drive')

hf_ime    = "models--" + model_name.replace("/", "--")
izvor     = f"/content/drive/MyDrive/hf_cache/hub/{hf_ime}"
odrediste = f"/content/hf_cache/hub/{hf_ime}"

if not os.path.exists(odrediste):
    print("Kopiranje modela na /content...")
    !mkdir -p /content/hf_cache/hub
    !cp -r {izvor} {odrediste}
    print("Gotovo.")

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print(f"Model: {model_name}")
print(f"Čitanje s /content: {os.environ['HF_HOME']}")

### Konfiguracija kvantizacije i učitavanje LLM modela

### **BitsAndBytesConfig**
Ovaj dio koda definira konfiguraciju za kvantizaciju modela koristeći klasu **`BitsAndBytesConfig`** iz biblioteke **`bitsandbytes`**, koja omogućava efikasnije korištenje memorije i ubrzanje modela prilikom izvršavanja.

- **`BitsAndBytesConfig`**: Ova klasa se koristi za konfiguraciju kvantizacije modela, što je tehnika koja smanjuje preciznost numeričkih podataka (npr. od 32-bitnih do 4-bitnih) kako bi se smanjila memorijska potrošnja i ubrzalo izvođenje modela, uz minimalan utjecaj na točnost.

- **`load_in_4bit=True`**: Ova opcija omogućava učitavanje modela u 4-bitnom formatu. To značajno smanjuje veličinu modela u memoriji, što je korisno za rad s velikim modelima na uređajima s ograničenim resursima, poput GPU-ova s manje VRAM-a.

- **`bnb_4bit_use_double_quant=True`**: Ova opcija aktivira *double quantization* (dvostruku kvantizaciju). To znači da se kvantizacija primjenjuje u dva koraka, što dodatno smanjuje memorijsku potrošnju i poboljšava efikasnost.

- **`bnb_4bit_compute_dtype=torch.float16`**: Ovdje se specificira tip podataka koji će se koristiti za računanje tijekom izvođenja modela. U ovom slučaju, koristi se `float16`, što je kompromis između preciznosti i efikasnosti. Ovaj tip podataka je posebno optimiziran za GPU-ove.

- **`bnb_4bit_quant_type="nf4"`**: Ova opcija određuje tip kvantizacije koji će se koristiti. `"nf4"` označava *Normal Float 4* (NF4), što je specifičan format kvantizacije koji je optimiziran za bolje performanse i točnost u poređenju s tradicionalnim 4-bitnim formatima.

Ova konfiguracija omogućava učitavanje modela u 4-bitnom formatu koristeći napredne tehnike kvantizacije (poput dvostruke kvantizacije i NF4 formata) kako bi se smanjila memorijska potrošnja i ubrzalo izvođenje modela, dok se istovremeno održava dovoljno visoka točnost. Ovo je posebno korisno za rad s velikim jezičkim modelima na uređajima s ograničenim resursima.

In [5]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

pipe = pipeline(
    "text-generation",
    model=model_name,
    dtype=torch.float16,
    device_map="auto",
    model_kwargs={"quantization_config": quantization_config}
)

pipe.model.generation_config.max_length = 8192

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/40.3k [00:00<?, ?B/s]

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.3k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/2.41M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

Opisano člankom na sljedećem linku:

[Making LLMs even more accessible with bitsandbytes, 4-bit quantization and QLoRA](https://huggingface.co/blog/4bit-transformers-bitsandbytes)

### Kreiranje pipeline-a za generaciju teksta

Pipeline kombinira model, tokenizer i konfiguraciju kvantizacije u jednostavan interfejs za generiranje teksta.

Ovaj dio koda koristi biblioteku **Hugging Face Transformers** za kreiranje *pipelinea* za generaciju teksta s pomoću unaprijed istreniranog jezičnog modela. Evo detaljnog objašnjenja:

- **`pipeline`**: Funkcija `pipeline` iz biblioteke Transformers omogućava jednostavno kreiranje unaprijed definiranih tokova obrade (kao što su generacija teksta, klasifikacija, prevođenje, itd.) koristeći unaprijed istrenirane modele.

- **`"text-generation"`**: Ovaj argument specificira da se koristi pipeline za generaciju teksta, što znači da će model generirati tekst na osnovu unosa (prompta) koji mu se da.

- **`model=model_name`**: Ovdje se specificira koji model će se koristiti. U ovom slučaju, koristi se model definiran varijablom `model_name` — trenutno *EuroLLM-22B-Instruct*, višejezični europski model s 22 milijarde parametara optimiziran za instrukcijske zadatke.

- **`dtype=torch.float16`**: Ova opcija postavlja tip podataka za tenzore na `float16`, što smanjuje memorijsku potrošnju i ubrzava obradu, posebno na GPU-ovima.

- **`device_map="auto"`**: Ova opcija automatski određuje na kojem uređaju će se model izvršavati (npr. CPU ili GPU). Ako je dostupan GPU, model će se automatski premjestiti na njega radi brže obrade.

- **`model_kwargs={"quantization_config": quantization_config}`**: Ovdje se prosljeđuju dodatni parametri modelu, konkretno konfiguracija kvantizacije koja je prethodno definirana.

[Pipeline](https://huggingface.co/docs/transformers/pipeline_tutorial)

## 3. Tokenizacija - demonstracija

Tokenizacija je proces dijeljenja teksta na manje jedinice - tokene. Svaki token je obično riječ, dio riječi ili specijalni znak. Tokenizator konvertuje tekstualne stringove u numeričke ID-ove koje model može obraditi.

### Učitavanje tokenizera

Učitavamo tokenizer za odabrani model. Za demonstraciju tokenizacije potreban je samo tokenizer, ne i cijeli model.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

[Tokenizer](https://huggingface.co/docs/transformers/en/main_classes/tokenizer)

### Primjer tokenizacije

Demonstriramo kako tokenizer rastavlja tekst na tokene i pretvara ih u numeričke ID-ove.

In [7]:
text = "S koliko godina časnički namjesnik ide u mirovinu?"
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokeni:", tokens)
print("Token ID-ovi:", token_ids)
print("Broj tokena:", len(tokens))

Tokeni: ['▁S', '▁koliko', '▁godina', '▁čas', 'nički', '▁nam', 'jes', 'nik', '▁ide', '▁u', '▁miro', 'vin', 'u', '?']
Token ID-ovi: [577, 67135, 22589, 7247, 81812, 3164, 3545, 1375, 2447, 703, 86708, 3526, 119726, 119882]
Broj tokena: 14


## 4. Generacija odgovora

Definiramo sistemsku poruku, funkciju za generaciju teksta i `vanilla_chat` funkciju koja šalje pitanje modelu bez RAG konteksta.

### Funkcija za generiranje odgovora

Funkcija `generate_response` šalje poruke modelu i vraća generirani odgovor. Koristi nisku temperaturu (0.3) za konzistentnije odgovore te u postprocesiranju uklanja nepotpune odgovore koji ne završavaju interpunkcijskim znakom.

### Definicija sistemske poruke (System Prompt)

Sistemska poruka postavlja ulogu i način ponašanja modela. Model je definiran kao pravni stručnjak za zakonodavstvo Republike Hrvatske s naglaskom na Zakon o obrani i Zakon o službi u Oružanim snagama. Poruka sadrži i kritične upute za sprječavanje generiranja odgovora na krivom jeziku ili pismu.

### Putanja CSV datoteke i Vanilla chat funkcija

Definiramo putanju CSV datoteke za pohranu svih rezultata testiranja, te funkciju `vanilla_chat` koja šalje pitanje modelu bez RAG konteksta. Ova funkcija koristi nisku temperaturu (0.3) za konzistentnije odgovore i eksplicitno traži odgovor na hrvatskom jeziku latinicom.

In [8]:
def generate_response(
    messages,
    max_new_tokens=512,
    temperature=0.5,
    do_sample=True,
):
    final_messages = list(messages)

    model_max_length = getattr(pipe.tokenizer, 'model_max_length', 8192)
    if model_max_length > 100000:
        model_max_length = 8192

    input_text = pipe.tokenizer.apply_chat_template(final_messages, tokenize=False)
    input_token_count = len(pipe.tokenizer.encode(input_text))
    max_allowed_input = model_max_length - max_new_tokens

    if input_token_count > max_allowed_input:
        print(f"Upozorenje: input ({input_token_count} tokena) prelazi limit ({max_allowed_input}).")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gen_config = deepcopy(pipe.model.generation_config)
    gen_config.max_new_tokens = max_new_tokens
    gen_config.temperature = temperature
    gen_config.do_sample = do_sample
    gen_config.repetition_penalty = 1.1

    vrijeme_upita = time.time()

    response = pipe(
        final_messages,
        generation_config=gen_config,
    )

    answer = ""
    for message in response[0]['generated_text']:
        if message['role'] == 'assistant':
            answer = message['content']
            break

    stripped_answer = answer.strip()
    if not re.search(r'[.!?]$', stripped_answer) and len(stripped_answer.split()) > 5:
        last_punctuation_index = -1
        for i in range(len(stripped_answer) - 1, -1, -1):
            if stripped_answer[i] in ['.', '?', '!']:
                last_punctuation_index = i
                break

        if last_punctuation_index != -1:
            answer = stripped_answer[:last_punctuation_index + 1]
        else:
            words = stripped_answer.split()
            if len(words) > 1:
                answer = ' '.join(words[:-1]) + '...'
            elif words:
                answer = words[0]
            else:
                answer = ''
    else:
        answer = stripped_answer

    vrijeme_odgovora = time.time()
    ukupno_vrijeme = vrijeme_odgovora - vrijeme_upita

    print("Generirani odgovor:")
    print(answer)
    print(f"Ukupno vrijeme za odgovor: {ukupno_vrijeme:.2f} sekundi")

    return answer, ukupno_vrijeme

In [9]:
SYSTEM_PROMPT = (
"Ti si pravni stručnjak specijaliziran za zakonodavstvo Republike Hrvatske, posebno za Zakon o obrani, Zakon o službi u Oružanim snagama RH i pripadajuće pravilnike."
"\n\nPravila odgovaranja:"
"\n1. Jezik odgovora: hrvatski, latinica."
"\n2. Navedi konkretne članke zakona, brojčane vrijednosti i nadležna tijela."
"\n3. Ako je dostupan kontekst, koristi SAMO informacije iz njega."
"\n4. Ako kontekst ne sadržava odgovor, odgovori na temelju općeg znanja i dodaj: 'Napomena: ova informacija nije pronađena u dostupnim dokumentima."
"\n5. Odgovor neka bude koncizan i strukturiran — bez nepotrebnih uvoda."
)

In [10]:
csv_path = '/content/drive/My Drive/diplomskiRad/rezultati.csv'

def vanilla_chat(user_question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{user_question}\n\nOdgovori NA HRVATSKOM JEZIKU koristeći LATINICU."}
    ]
    # Izjednaceni generacijski parametri (Vanilla = RAG = GraphRAG):
    # 512 tokena i deterministicko dekodiranje radi usporedivosti
    return generate_response(messages, max_new_tokens=512, temperature=0.3, do_sample=False)

## 5. Testiranje — Vanilla

Vanilla pristup koristi LLM model bez dodatne baze znanja. Model se oslanja isključivo na znanje stečeno tijekom treniranja. Ovo služi kao bazna linija za usporedbu s RAG pristupom.

### Učitavanje pitanja i odgovora iz JSON datoteke

Učitavamo testna pitanja i očekivane odgovore iz JSON datoteke. Q&A parovi koriste se isključivo za testiranje — ne dodaju se u bazu znanja.

In [11]:
try:
    with open('/content/drive/My Drive/diplomskiRad/pitanja_odgovori.json', 'r', encoding='utf-8') as f:
        qa_data = json.load(f)

    print(f"✓ Učitano {qa_data['metadata']['ukupno_pitanja']} pitanja i odgovora.")
    print(f"  Struktura:")
    for kategorija, info in qa_data['metadata']['struktura'].items():
        print(f"    - {kategorija}: {info['broj']} ({info['postotak']})")
except FileNotFoundError:
    print('Datoteka pitanja_odgovori.json nije pronađena na navedenoj putanji.')
    qa_data = None
except json.JSONDecodeError:
    print('Greška pri parsiranju JSON datoteke. Provjerite je li format ispravan.')
    qa_data = None

✓ Učitano 43 pitanja i odgovora.
  Struktura:
    - kompleksna_vise_dokumenata: 12 (12.9%)
    - kompleksna_jedan_dokument: 8 (12.9%)
    - jednostavna: 23 (74.2%)


### Testiranje Vanilla pristupa - sva pitanja

Prolazimo kroz sva pitanja iz JSON datoteke, generiramo odgovore bez RAG-a i spremamo rezultate u CSV datoteku.

In [12]:
if qa_data is None:
    print("QA datoteka nije ucitana.")
else:
    all_questions = []
    all_categories = []
    for key, value in qa_data.items():
        if key != 'metadata' and isinstance(value, list):
            for item in value:
                all_questions.append(item)
                all_categories.append(key)

    vanilla_rezultati = []

    for i, (qa_item, kategorija) in enumerate(zip(all_questions, all_categories)):
        user_question = qa_item.get('pitanje', qa_item.get('question', ''))
        ocekivani = qa_item.get('odgovor', qa_item.get('answer', ''))

        print(f"\n{'='*80}")
        print(f"Pitanje {i + 1}/{len(all_questions)}: {user_question}")
        print(f"Kategorija: {kategorija}")
        print(f"{'='*80}")

        answer, vrijeme = vanilla_chat(user_question)

        vanilla_rezultati.append({
            'rbr': i + 1,
            'kategorija': kategorija,
            'pitanje': user_question,
            'ocekivani_odgovor': ocekivani,
            'vanilla_odgovor': answer,
            'vanilla_vrijeme_sekunde': round(vrijeme, 2)
        })

    df_rezultati = pd.DataFrame(vanilla_rezultati)
    df_rezultati.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"\n{'='*80}")
    print(f"Vanilla rezultati spremljeni u: {csv_path}")
    print(f"Ukupno obradeno pitanja: {len(vanilla_rezultati)}")
    print(f"Prosjecno vrijeme po pitanju: {df_rezultati['vanilla_vrijeme_sekunde'].mean():.2f} sekundi")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Pitanje 1/43: Opišite cjelokupnu proceduru transformacije novaka u razvrstanog pričuvnika kroz sve pravne akte koji reguliraju taj proces.
Kategorija: kompleksna_pitanja_vise_dokumenata


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generirani odgovor:
Transformacija novaka u razvrstani pričuvnik u Hrvatskoj regulirana je Zakonom o službi u Oružanim snagama Republike Hrvatske (ZSOŠ) te pripadajućim pravilnicima. Evo korak-po-korak procedure:  

1. **Raspored na odsluženje vojnog roka** – Prema članku 60. ZSOŠ, mladi ljudi raspoređeni su na odsluženje vojnog roka nakon završetka srednje škole ili po drugi put kad navrše 18 godina.  

2. **Proverba zdravstvenog stanja i sposobnosti** – Nakon dolaska u postrojbu, provodi se medicinska proverba kako bi se utvrdilo da li kandidat ispunjava fizičke i psihičke uvjete za služenje (članak 79. ZSOŠ).  

3. **Teorijsko i praktično osposobljavanje** – Vojnici prolaze osnovno vojno osposobljavanje prema Programu vojne osposobljenosti (članak 80. ZSOŠ), uključujući topografiju, taktiku, uporabu oružja itd.  

4. **Ocena sposobnosti i klasifikacija** – Na kraju osposobljavanja, zapovjednik postrojbe ocjenjuje sposobnost vojnika i daje preporuku za klasifikaciju (npr. "odlično", 

[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Generirani odgovor:
Minimalno trajanje godišnjeg odmora djelatne vojne osobe određeno je člankom **80.** Zakona o službi u Oružanim snagama RH (Narodne novine br. 97/16). Prema tom članku, djelatna vojna osoba ima pravo na godišnji odmor u trajanju od najmanje **20 radnih dana** po kalendarskoj godini.  

Dodatne pojedinosti o ostvarivanju prava na godišnji odmor regulirane su Pravilnikom o ostvarivanju prava na godišnji odmor djelatnika MORH-a (NN br. 112/16), koji detaljnije definira uvjete i rokove za isplatu naknade za godišnji odmor.  

Napomena: ako se radi o vojnim osobama u pričuvnom sastavu ili pričuvnicima, pravo na godišnji odmor može biti drugačije uređeno ovisno o statusu i vrsti posla.
Ukupno vrijeme za odgovor: 26.18 sekundi

Pitanje 41/43: Koliko dana plaćenog dopusta pripada djelatnoj vojnoj osobi u slučaju rođenja djeteta?
Kategorija: jednostavna_pitanja


[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generirani odgovor:
Djelatna vojna osoba ima pravo na **plaćeni roditeljski dopust** od **8 tjedana** nakon rođenja djeteta (članak 70. Pravilnika o ostvarivanju prava na roditeljski dopust).  

Ako se radi o aktivnom sudioniku u službi (npr. časniku), dodatno može imati pravo na dodatni dopust prema članku 69. istog Pravilnika. Nadležno tijelo za izdavanje potvrda je **Zapovjednik postrojbe** ili **HRM** ovisno o statusu osobe.  

Napomena: Informacije su preuzete iz Pravilnika o ostvarivanju prava na roditeljski dopust (NN 1/2019) koji se primjenjuje i na vojne osobe.
Ukupno vrijeme za odgovor: 21.49 sekundi

Pitanje 42/43: Koliko dana plaćenog dopusta pripada djelatnoj vojnoj osobi u slučaju zaključenja braka?
Kategorija: jednostavna_pitanja


[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generirani odgovor:
Djelatna vojna osoba ima pravo na **7 dana** plaćenog dopusta u slučaju zaključenja braka, sukladno članku **80. stavku 1. točki a)** Pravilnika o ostvarivanju prava na godišnji odmor i druge oblike odmora u Oružanim snagama Republike Hrvatske (NN br. 9/16).  

Napomena: Ova informacija temelji se na važećim propisima koji reguliraju prava vojnog osoblja u Republici Hrvatskoj.
Ukupno vrijeme za odgovor: 13.90 sekundi

Pitanje 43/43: Što je vojna iskaznica i tko ju izdaje?
Kategorija: jednostavna_pitanja
Generirani odgovor:
Vojna iskaznica (VI) je osobna iskaznica vojnog obveznika koja potvrđuje njegovo pravo na služenje vojne službe te sadrži podatke o obveznim vojnim rokovanjima i statusu u sustavu Oružanih snaga Republike Hrvatske.  

**Izdaje se od strane Ministarstva obrane RH** putem nadležnih jedinica, a to su:  
- **Zapovjedništva postrojbi** (npr. pješačke brigade, topničke brigade itd.)  
- **Regionalni centri za obuku i mobilizaciju**  
- **Središnja uprav